In [0]:
%pip install confluent_kafka

In [0]:
from confluent_kafka import Producer
import json
from  itertools import islice
import numpy as np
from pyspark.sql.functions import col, decode, split, element_at, udf, lit, reduce, from_json, regexp_replace, concat, when
import logging
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType
import datetime
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkContext
import os
from functools import reduce
import time
import traceback

In [0]:
logger = logging.getLogger("DatabricksWorkflow")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
if not logger.hasHandlers():
    logger.addHandler(handler)

In [0]:
config_path = "dbfs:/configs/config.json"
try:
    config = spark.read.option("multiline", "true").json(config_path)
    logger.info(f"Successfully read config file from {config_path}")
except Exception as e:
    logger.error(f"Could not read config file at {config_path}: {e}", exc_info=True)
    raise FileNotFoundError(f"Could not read config file at {config_path}: {e}")

try:
    first_row = config.first()
    env = first_row["env"].strip().lower()
    lz_key = first_row["lz_key"].strip().lower()
    logger.info(f"Extracted configs: env={env}, lz_key={lz_key}")
except Exception as e:
    logger.error(f"Missing expected keys 'env' or 'lz_key' in config file: {e}", exc_info=True)
    raise KeyError(f"Missing expected keys 'env' or 'lz_key' in config file: {e}")

try:
    keyvault_name = f"ingest{lz_key}-meta002-{env}"
    logger.info(f"Constructed keyvault name: {keyvault_name}")
except Exception as e:
    logger.error(f"Error constructing keyvault name: {e}", exc_info=True)
    raise ValueError(f"Error constructing keyvault name: {e}")


In [0]:

try:
    client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-SECRET from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}")

try:
    tenant_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-TENANT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}")

try:
    client_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}")

logger.info("✅ Successfully retrieved all Service Principal secrets from Key Vault")


In [0]:
# --- Parameterise containers ---
curated_storage_account = f"ingest{lz_key}curated{env}"
curated_container = "gold"
silver_curated_container = "silver"
checkpoint_storage_account = f"ingest{lz_key}xcutting{env}"

# --- Assign OAuth to storage accounts ---
storage_accounts = [curated_storage_account, checkpoint_storage_account]

for storage_account in storage_accounts:
    try:
        configs = {
            f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
            f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
                "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
            f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
            f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
            f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
                f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
        }

        for key, val in configs.items():
            try:
                spark.conf.set(key, val)
            except Exception as e:
                logger.error(f"Failed to set Spark config '{key}' for storage account '{storage_account}': {e}", exc_info=True)
                raise RuntimeError(f"Failed to set Spark config '{key}' for storage account '{storage_account}': {e}")

        logger.info(f"✅ Successfully configured OAuth for storage account: {storage_account}")

    except Exception as e:
        logger.error(f"Error configuring OAuth for storage account '{storage_account}': {e}", exc_info=True)
        raise RuntimeError(f"Error configuring OAuth for storage account '{storage_account}': {e}")


In [0]:
ccd_case_link_call_result = spark.read.format("delta").load(f"abfss://silver@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CASE_LINKING/ack_audit")
ccd_case_link_call_result.createOrReplaceTempView("ccd_case_link_call_result")

ccd_case_link_publish_payload_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/CASE_LINKING/publish_payload_audit")
ccd_case_link_publish_payload_result.createOrReplaceTempView("ccd_case_link_publish_payload_result")

spark.sql("""
    SELECT 
        COALESCE(t2.RunId, t1.RunId) AS RunID,
        COALESCE(t2.CCDCaseReferenceNumber, t1.CCDCaseReferenceNumber) as `CCD Case Reference Number`,
        t1.Status as `Case Link Publish Status`,
        t2.Status as `Case Link CCD Call Status`,
        t1.CaseLinkCount as `Case Link Publish Case Link Count`,
        t1.StartDateTime as `Case Link Publish Publishing Date Time`,
        t1.Error as `Case Link Publish Error`,
        t2.CaseLinkCount as `Case Link CCD Call Function App Case Link Count`,
        t2.StartDateTime as `Case Link CCD Call Function App Start Date Time`,
        t2.EndDateTime as `Case Link CCD Call Function App End Date Time`
        t2.Error as `Case Link CCD Call Function App Error`
    FROM ccd_publish_payload_result t1
        FULL OUTER JOIN ccd_call_result t2 ON t1.CCDCaseReferenceNumber = t2.CCDCaseReferenceNumber AND t1.RunID = t2.RunID
    ORDER BY t1.PublishingDateTime DESC
""").display()


Databricks visualization. Run in Databricks to view.

In [ ]:
case_link_groups = spark.read.table(f"hive_metastore.ariadm_active_appeals.silver_case_link_groups")
case_link_groups.createOrReplaceTempView("case_link_groups")
case_link_mappings = spark.read.table(f"hive_metastore.ariadm_active_appeals.aria_ccd_case_mappings")
case_link_mappings.createOrReplaceTempView("case_link_mappings")
case_link_combinations = spark.read.table(f"hive_metastore.ariadm_active_appeals.case_link_combinations")
case_link_combinations.createOrReplaceTempView("case_link_combinations")
case_link_payloads = spark.read.table(f"hive_metastore.ariadm_active_appeals.case_link_payloads")
case_link_payloads.createOrReplaceTempView("case_link_payloads")

ccd_case_link_call_result = spark.read.format("delta").load(f"abfss://silver@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CASE_LINKING/ack_audit")
ccd_case_link_call_result.createOrReplaceTempView("ccd_case_link_call_result")

ccd_case_link_publish_payload_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/CASE_LINKING/publish_payload_audit")
ccd_case_link_publish_payload_result.createOrReplaceTempView("ccd_case_link_publish_payload_result")

spark.sql("""
    SELECT 
        g.CaseNo AS CaseNo,
        p.CCDCaseReferenceNumberFrom AS `CCD Case Reference Number`,
        CASE WHEN g.CaseNo IS NOT NULL
            THEN 'YES'
            ELSE 'NO'
        END AS `ARIA Case Link Groups Exist`,
        CASE WHEN m.CaseNo IS NOT NULL AND m.CCDCaseReferenceNumber IS NOT NULL AND m.CCDCaseReferenceNumber = t2.CCDCaseReferenceNumber
            THEN 'SUCCESS'
            ELSE 'ERROR'
        END AS `ARIA-CCD Case Mappings Status`,
        CASE WHEN c.CCDCaseReferenceNumberFrom IS NOT NULL AND c.CCDCaseReferenceNumberFrom = t2.CCDCaseReferenceNumber
            THEN 'SUCCESS'
            ELSE 'ERROR'
        END AS `Case Link Combinations Status`,
        CASE WHEN p.CCDCaseReferenceNumberFrom IS NOT NULL AND p.CCDCaseReferenceNumberFrom != '' THEN 'SUCCESS' ELSE 'ERROR' END AS `Case Link Payloads Status`,
        t1.Status AS `Case Link Publish Status`,
        t2.Status AS `Case Link CCD Call Status`,
        COUNT(*) OVER (PARTITION BY g.LinkNo) - 1 AS `ARIA Case Link Groups Case Link Count`,
        t1.CaseLinkCount as `Case Link Publish Case Link Count`,
        t1.StartDateTime as `Case Link Publish Publishing Date Time`,
        t1.Error as `Case Link Publish Error`,
        t2.CaseLinkCount as `Case Link CCD Call Function App Case Link Count`,
        t2.StartDateTime as `Case Link CCD Call Function App Start Date Time`,
        t2.EndDateTime as `Case Link CCD Call Function App End Date Time`
        t2.Error as `Case Link CCD Call Function App Error`
    FROM case_link_groups g
        FULL OUTER JOIN case_link_mappings m ON g.CaseNo = m.CaseNo
        FULL OUTER JOIN case_link_combinations c ON m.CaseNo = c.CaseNoFrom
        FULL OUTER JOIN case_link_payloads p ON c.CCDCaseReferenceNumberFrom = p.CCDCaseReferenceNumberFrom
        FULL OUTER JOIN ccd_publish_payload_result t1 ON p.CCDCaseReferenceNumberFrom = t1.CCDCaseReferenceNumber
        FULL OUTER JOIN ccd_call_result t2 ON t1.CCDCaseReferenceNumber = t2.CCDCaseReferenceNumber
    ORDER BY t1.PublishingDateTime DESC
""").display()


In [0]:
dbutils.notebook.exit("Notebook completed successfully")